# Proje Amacı

Bu proje, bir metnin bir insan tarafından mı yazıldığını yoksa bir Büyük Dil Modeli (LLM) tarafından mı oluşturulduğunu ayırt etmeyi amaçlıyor.

In [52]:
# Veri manipülasyonu ve analizi için kullanılır; tabloları (DataFrame) okumayı ve düzenlemeyi sağlar.
import pandas as pd

# Bilimsel hesaplamalar ve çok boyutlu diziler (matrix) üzerinde hızlı matematiksel işlemler yapmak içindir.
import numpy as np

# Verilerin temel 2 boyutlu grafiklerini (çizgi, sütun, dağılım vb.) oluşturmak için kullanılan temel kütüphanedir.
import matplotlib.pyplot as plt

# Matplotlib tabanlı, daha gelişmiş, estetik ve istatistiksel veri görselleştirmeleri yapmak için kullanılır.
import seaborn as sns

#uyarı görmemek için
import warnings
warnings.filterwarnings('ignore')

In [53]:
#veriyi oku
train=pd.read_csv('train_v2_drcat_02.csv').copy()
train_prompts=pd.read_csv('train_prompts.csv').copy()
test=pd.read_csv('test_essays.csv').copy()

In [54]:
#ilk 5 satırı gör
train.head()

,text,label,prompt_name,source,RDizzl3_seven
0,Phones\n\nModern humans today are always on th...,0,Phones and driving,persuade_corpus,False
1,This essay will explain if drivers should or s...,0,Phones and driving,persuade_corpus,False
2,Driving while the use of cellular devices\n\nT...,0,Phones and driving,persuade_corpus,False
3,Phones & Driving\n\nDrivers should not be able...,0,Phones and driving,persuade_corpus,False
4,Cell Phone Operation While Driving\n\nThe abil...,0,Phones and driving,persuade_corpus,False


In [55]:
test.head()

,id,prompt_id,text
0,0000aaaa,2,Aaa bbb ccc.
1,1111bbbb,3,Bbb ccc ddd.
2,2222cccc,4,CCC ddd eee.


In [56]:
#son 5 satırı göster
train.tail()

,text,label,prompt_name,source,RDizzl3_seven
44863,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44864,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44865,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44866,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True
44867,"Dear Senator,\n\nI am writing to you today to ...",1,Does the electoral college work?,kingki19_palm,True


In [57]:
train_prompts.head()

,prompt_id,prompt_name,instructions,source_text
0,0,Car-free cities,Write an explanatory essay to inform fellow ci...,"# In German Suburb, Life Goes On Without Cars ..."
1,1,Does the electoral college work?,Write a letter to your state senator in which ...,# What Is the Electoral College? by the Office...


In [58]:
train['label'].unique()

array([0, 1])

In [59]:
test_ids=test['id']
test_prompt=test['prompt_id']

In [60]:
#text ve generated sütunlarını seçiyoruz
train_df=train[['text', 'label']]

In [61]:
train_df.head()

,text,label
0,Phones\n\nModern humans today are always on th...,0
1,This essay will explain if drivers should or s...,0
2,Driving while the use of cellular devices\n\nT...,0
3,Phones & Driving\n\nDrivers should not be able...,0
4,Cell Phone Operation While Driving\n\nThe abil...,0


In [62]:
#1000.veriyi göster
train_df['text'][1000]

"Distractions while driving could lead to deaths of other civilians. There are reasons why this topic became an actual law. The law was just passed last year that is you're on your phone while driving, that is an immediate ticket. Due to your criminal record (if it's bad) you can even do jail time if you feel you do not apply to this law. The cops of our cities are doing their job by stopping people who does not follow this law.\n\nThe main reason why this show not be condoned is, because most of these situations a citizen could get in a car accident. Approximately 1.6 million citizens die due to being on their cell phone while operating a vehicle. Nearly 390,000 people get real serious injuries due to texting and driving. One out of four of every car accident in the United States are due to texting and driving. In my opinion I believe texting and driving is the main reason we have so many deaths every year.\n\nIn conclusion, texting and driving needs to be/ stay illegal. That way peop

In [63]:
#pip install neattext

In [64]:
import neattext.functions as nfx

In [65]:
def clean_text(text):
    text=text.lower()
    text=nfx.remove_special_characters(text)
    text=nfx.remove_multiple_spaces(text)
    return text

In [66]:
train_df['text']=train_df['text'].apply(clean_text)

In [67]:
#1000.veriyi göster
train_df['text'][1000]

'distractions while driving could lead to deaths of other civilians there are reasons why this topic became an actual law the law was just passed last year that is youre on your phone while driving that is an immediate ticket due to your criminal record if its bad you can even do jail time if you feel you do not apply to this law the cops of our cities are doing their job by stopping people who does not follow this lawthe main reason why this show not be condoned is because most of these situations a citizen could get in a car accident approximately 16 million citizens die due to being on their cell phone while operating a vehicle nearly 390000 people get real serious injuries due to texting and driving one out of four of every car accident in the united states are due to texting and driving in my opinion i believe texting and driving is the main reason we have so many deaths every yearin conclusion texting and driving needs to be stay illegal that way people will not do it so often wh

In [68]:
#x ve y ayırma
x = train_df['text']       # Bağımsız değişken: Metin verisi
y = train_df['label']  # Bağımsız değişken: 0 (İnsan) veya 1 (AI)

In [69]:
from textblob import TextBlob

def lemmatize_text(text):
    words = TextBlob(text).words
    return [word.lemmatize() for word in words]

In [70]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [71]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [72]:
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer(ngram_range=(1,2), stop_words='english', max_features=5000)

# Sadece 'x' değişkenindeki metinleri sayıya dönüştür
x_vect = vect.fit_transform(x).toarray()

In [73]:
from sklearn.model_selection import train_test_split

# x_vect (sayısal metinler) ve y (etiketler) kullanılarak bölünür
x_train, x_test, y_train, y_test = train_test_split(x_vect, y, test_size=0.2, random_state=42)

In [74]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(128, activation='relu', input_shape=(x_vect.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid') # 1: AI, 0: İnsan (Binary Classification)
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [75]:
model.fit(x_train,y_train,batch_size=32, validation_data=(x_test,y_test), verbose=2, epochs=10)  #verbose=1 bütün epochların sonucunu gösterir, verbose=0 yazarsan sadece sonucu gösterir, 2 yazarsan daha detaylı bilgi verir

Epoch 1/10
1122/1122 - 15s - 13ms/step - accuracy: 0.9847 - loss: 0.0452 - val_accuracy: 0.9893 - val_loss: 0.0342
Epoch 2/10
1122/1122 - 19s - 17ms/step - accuracy: 0.9972 - loss: 0.0093 - val_accuracy: 0.9914 - val_loss: 0.0295
Epoch 3/10
1122/1122 - 12s - 11ms/step - accuracy: 0.9985 - loss: 0.0056 - val_accuracy: 0.9936 - val_loss: 0.0262
Epoch 4/10
1122/1122 - 12s - 11ms/step - accuracy: 0.9990 - loss: 0.0030 - val_accuracy: 0.9933 - val_loss: 0.0319
Epoch 5/10
1122/1122 - 12s - 11ms/step - accuracy: 0.9993 - loss: 0.0031 - val_accuracy: 0.9933 - val_loss: 0.0327
Epoch 6/10
1122/1122 - 21s - 18ms/step - accuracy: 0.9993 - loss: 0.0022 - val_accuracy: 0.9931 - val_loss: 0.0310
Epoch 7/10
1122/1122 - 12s - 11ms/step - accuracy: 0.9994 - loss: 0.0026 - val_accuracy: 0.9930 - val_loss: 0.0375
Epoch 8/10
1122/1122 - 12s - 11ms/step - accuracy: 0.9997 - loss: 9.5366e-04 - val_accuracy: 0.9941 - val_loss: 0.0443
Epoch 9/10
1122/1122 - 12s - 11ms/step - accuracy: 0.9997 - loss: 9.8324e-04

In [76]:
model.evaluate(x_test, y_test)  #modelin basari orani

281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9939 - loss: 0.0291


[0.029074328020215034, 0.9938712120056152]

In [77]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Modelden tahminleri al
y_pred = model.predict(x_test)
y_pred_binary = (y_pred > 0.5).astype(int)

# Karmaşıklık matrisini yazdır
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_binary))

# Detaylı rapor
print("\nClassification Report:")
print(classification_report(y_test, y_pred_binary))

281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Confusion Matrix:
[[5466   15]
 [  40 3453]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      5481
           1       1.00      0.99      0.99      3493

    accuracy                           0.99      8974
   macro avg       0.99      0.99      0.99      8974
weighted avg       0.99      0.99      0.99      8974



In [78]:
train_df.sample(10)

,text,label
36553,dear senator i am writing to express my opinio...,1
41599,hey everyonewe all love our cars but lets fac...,1
24749,you have a big decision to make the stress is ...,0
28839,as an eighthgrade student i have heard the quo...,1
25546,everyone has asked for advice or asked for som...,0
5200,you should participate in the seagoing cowboys...,0
35585,seeking multiple opinions when asking for advi...,1
11837,i dont beleive that the use of this technology...,0
39459,in the vast and diverse world we live in today...,1
1742,cars they are a part of alot of peoples everyd...,0


In [79]:
train_df['text'][43062]

'a life filled to the brim of better days is what we all want and limiting car usage has some of these advantages when i had a car i was always tense im much happier this way said heidrun walter the sound of children outside of the pollution free community is much more satisfying then getting somewhere with dangers involved a life without cars is less stressful and more free to explore in safety driving tends to make drivers uneasy for fear or crash or being late to work from trafficmr sivaks son lives in san francisco and has a car but takes bay area rapid transit when he can even though that often takes longer than driving clearly being late to work and stressed is not something people favor epically when getting to work for free is possible public transit was free of charge from friday to monday according to the bbc instead of spending thousands of dollars expecting an easier lifestyle your neck rests on the chopping block with the blade ever silently above until it completes is mis

In [80]:
test1=['a life filled to the brim of better days is what we all want and limiting car usage has some of these advantages when i had a car i was always tense im much happier this way said heidrun walter the sound of children outside of the pollution free community is much more satisfying then getting somewhere with dangers involved a life without cars is less stressful and more free to explore in safety driving tends to make drivers uneasy for fear or crash or being late to work from trafficmr sivaks son lives in san francisco and has a car but takes bay area rapid transit when he can even though that often takes longer than driving clearly being late to work and stressed is not something people favor epically when getting to work for free is possible public transit was free of charge from friday to monday according to the bbc instead of spending thousands of dollars expecting an easier lifestyle your neck rests on the chopping block with the blade ever silently above until it completes is mission having this ability feels like freedom but its actually just a fear inducing death causing machine a simple healthier life on a bike seems more enjoyable and even more so when it promotes good health and your lifein this new approach stores are placed a walk away on a main street rather than in malls along some distant highway not one person can really enjoy a twenty or forty minute drive to a mall or anywhere for that matter its a waste of time money and life taking a 5 minute walk to a mall is much more beneficial to your body and health than adding gas to your lungs and its quicker too parks and sports centers also have bloomed throughout the city uneven pitted sidewalks have been replaced by broad smooth sidewalks rushhour restrictions have dramatically cut traffic and new restaurants and upscale shopping districts have cropped up children playing outside instead of inside on their new tablet could be our future again parents dont have enough time to spend thirty minutes driving to a park when their child seems content on their fancy iphone 6 shopping is also a favorite anywhere but it seems to be drifting online taking five minutes to walk to the mall and save money on not buying a car or spending money on shipping will give more money to buy stuff a bonus is no more wrong clothing sizes or ordering a lotion that smells like toilet water ease of lifestyle is a desire that can be fulfilled when waving goodbye to miles of road and hello to a new jeans that fit for oncepollution is a horrible topic that everyone wants to stop however everyone is ignoring one of its sources after days of nearrecord pollution paris enforced a partial driving ban to clear the air of the global city it sounds like earth is becoming like the movie walle and next stop is a ship in space leaving this trashed planet behind paris is setting a great example at trying to remove pollution for healthier living and a better environment the united states has clearly been challanged and in its effort to be the best it followed suit it will have beneficial implications for carbon emissions and the evironment it is hard to imagine a healthy ecosystem with flourishing plant and animal life in our current situation with paris and the united states assisting in the idea of other transpot methods not only will animals have a better chance at not becoming extinct but so will the human race dying of pollution will no longer be a concern and money spent on poison can go to lifes other pleasureshappiness fast travel no pollution and more new stuff sounds like owning a car right carlos arturo has a taste of the sweet life its a good opportunity to take away stress and lower air pollution sadly this is not what a car provides because in fact it only reverses these good things despair sets in with forty minutes waiting in traffic while clouds of gray swirl into the sky returning a shirt that didnt fit reconsidering the advantages of having a car and considering the advantages of limiting car usage is the right way to live a perfect life']

In [81]:
train_df['text'][39978]

'hey so for this essay i was like totally stoked to write about how seeking guidance from experts and authorities can be super helpful in life i mean who doesnt love getting advice from people who know what theyre talking about rightso like i started thinking about times in my life when ive sought guidance from people who are like really smart or experienced in something for example when i was trying to decide which college to apply to my parents were like totally helpful they asked me what i wanted to study and what kind of career i wanted and they helped me research different schools that would be a good fit and like it was really helpful because they knew way more about colleges than i didbut its not just parents right i mean there are tons of other people who can give good advice like my coach is super smart about sports and stuff so when im trying to improve my game i go to him for tips and my friends theyre like really good at some things so when im trying to decide on a movie to

In [82]:
test2=['hey so for this essay i was like totally stoked to write about how seeking guidance from experts and authorities can be super helpful in life i mean who doesnt love getting advice from people who know what theyre talking about rightso like i started thinking about times in my life when ive sought guidance from people who are like really smart or experienced in something for example when i was trying to decide which college to apply to my parents were like totally helpful they asked me what i wanted to study and what kind of career i wanted and they helped me research different schools that would be a good fit and like it was really helpful because they knew way more about colleges than i didbut its not just parents right i mean there are tons of other people who can give good advice like my coach is super smart about sports and stuff so when im trying to improve my game i go to him for tips and my friends theyre like really good at some things so when im trying to decide on a movie to watch or a book to read i ask them for recommendationsand its not just about like personal stuff there are also experts in different fields who can give really valuable advice like if youre interested in science or history or something you can go to a professor or a scientist and they can teach you way more than you could ever learn on your ownbut like the thing is you have to be willing to listen and take their advice if youre not willing to like open your mind and consider other peoples ideas then youre not going to learn anything and thats why its important to be brave and make your own decisions but also to be willing to ask for help when you need itand like i know some people might be thinking well what about when the experts are wrong or what about when their advice doesnt work and thats a good point but i think its important to remember that no one knows everything and even experts can make mistakes so like its okay to question their advice']

In [83]:
train_df['text'][19469]

'the electoral collage is a process that helps use vote for the preaident and vice president and they count the electores votes and it is desited by congress and most states have a winner takes all system sorce federal register most people dont like the electoral collage process most people prefer the derect election insted and over 60 percent prefer it and according to a gallup pull in 2000 taken shortly after algore thanks to the electoral collage he won the popular vote but lost the presidensy sorce federal register a argumant over the outcome of the electoral collage vote has happend in the past there was one in 2000 but thay dont think it will happen eny time soon and the only reson the electoral collage votes exseeds is becase of the popular votes for exampal in 2012 obam receved 617 percant of the electoral votes when he only had 513 of the populare votes because they based it on a winner takes all vote sorce federal register the electoral collage vote hast to have a presidentia

In [84]:
test3=['the electoral collage is a process that helps use vote for the preaident and vice president and they count the electores votes and it is desited by congress and most states have a winner takes all system sorce federal register most people dont like the electoral collage process most people prefer the derect election insted and over 60 percent prefer it and according to a gallup pull in 2000 taken shortly after algore thanks to the electoral collage he won the popular vote but lost the presidensy sorce federal register a argumant over the outcome of the electoral collage vote has happend in the past there was one in 2000 but thay dont think it will happen eny time soon and the only reson the electoral collage votes exseeds is becase of the popular votes for exampal in 2012 obam receved 617 percant of the electoral votes when he only had 513 of the populare votes because they based it on a winner takes all vote sorce federal register the electoral collage vote hast to have a presidential canidet that hast to have no transregional appeal because the canident with only regonal appeals is unlikely to be a sucssesful prasident and the residents of the other regions are more likly to feel that there votes dont count sorce federal register the electoral vote helps balens out the biger states votes like in florida when obama won with 29 electoral votes but its proven that a biger state gets more atention than a smaller state because it gets more attention from presidential canidets dering campaning sorce federal registerthe electoral vote avoids problems in some cases were a canidet dosent get a mager vote and the reson they have the electoral vote is so wiel the elections are going on a canodet dusent drop out and thay do this so thay can have a clear winner sorce fedral registerthe elector collage vote is regarded as a nondemocratic methido selecting a president and when you are voting for a presidentioal canodent you are actoly voting for a slate of electors and it is posible that the winner of the elector vote will not win the populer vote sorce richared a posner ']

In [85]:
train_df['text'][26917]

'hey yall today were gonna talk about how having a positive attitude is super important for success in life like its like totally crucial so first off having a positive attitude helps you stay motivated and focused on your goals when youre positive youre more likely to take risks and try new things which can lead to bigger and better opportunities plus its way easier to bounce back from setbacks when you have a positive mindset like if you fail at something youre more likely to be like oh well no big deal ill just try again instead of getting all down on yourself and giving up and like its not just about you and your goals man having a positive attitude can totally change your social life and relationships too when youre positive and optimistic youre way more fun to be around and people are more likely to want to hang out with you and be friends with you plus its way easier to resolve conflicts and stuff when youre being all positive and stuff but like the thing is its not always easy 

In [86]:
test4=['hey yall today were gonna talk about how having a positive attitude is super important for success in life like its like totally crucial so first off having a positive attitude helps you stay motivated and focused on your goals when youre positive youre more likely to take risks and try new things which can lead to bigger and better opportunities plus its way easier to bounce back from setbacks when you have a positive mindset like if you fail at something youre more likely to be like oh well no big deal ill just try again instead of getting all down on yourself and giving up and like its not just about you and your goals man having a positive attitude can totally change your social life and relationships too when youre positive and optimistic youre way more fun to be around and people are more likely to want to hang out with you and be friends with you plus its way easier to resolve conflicts and stuff when youre being all positive and stuff but like the thing is its not always easy to be positive you know life can be super hard sometimes and its easy to get all down and stuff but like the more you practice being positive the easier it gets and like its totally worth it man being positive can totally change your life so yeah thats my essay on why having a positive attitude is important for success in life and social relationships its like totally true man give it a try and see for yourself ']

In [87]:
#  Metni vectorizer ile sayısal diziye (array) çevirelim
# Not: Sadece 'transform' kullanıyoruz, 'fit_transform' değil.
cleaned_test3 = clean_text(test3[0]) # test3 bir liste olduğu için ilk elemanını alıyoruz
test3_vect = vect.transform([cleaned_test3]).toarray()

# Modelden tahmin alalım
prediction_prob = model.predict(test3_vect)

# Sonucu yazdıralım
labels = ['İnsan Yazısı (Human)', 'Yapay Zeka Yazısı (AI)']
result = labels[1 if prediction_prob > 0.5 else 0]

print(f"AI Olma Olasılığı: %{prediction_prob[0][0] * 100:.2f}")
print(f"Tahmin Sonucu: {result}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
AI Olma Olasılığı: %0.00
Tahmin Sonucu: İnsan Yazısı (Human)


In [88]:
#  Metni vectorizer ile sayısal diziye (array) çevirelim
# Not: Sadece 'transform' kullanıyoruz, 'fit_transform' değil.
cleaned_test2 = clean_text(test2[0]) # test2 bir liste olduğu için ilk elemanını alıyoruz
test2_vect = vect.transform([cleaned_test2]).toarray()

# Modelden tahmin alalım
prediction_prob = model.predict(test2_vect)

# Sonucu yazdıralım
labels = ['İnsan Yazısı (Human)', 'Yapay Zeka Yazısı (AI)']
result = labels[1 if prediction_prob > 0.5 else 0]

print(f"AI Olma Olasılığı: %{prediction_prob[0][0] * 100:.2f}")
print(f"Tahmin Sonucu: {result}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
AI Olma Olasılığı: %100.00
Tahmin Sonucu: Yapay Zeka Yazısı (AI)


In [89]:
import joblib

# 1. Modeli Kaydet
model.save('llm_detector_model.keras')

# 2. Vektörleştiriciyi Kaydet
# Bu çok önemli, çünkü tahmin yaparken kelimeleri aynı sayılara çevirmemiz gerek.
joblib.dump(vect, 'vectorizer.pkl')

print("Model ve Vektörleştirici başarıyla paketlendi!")

Model ve Vektörleştirici başarıyla paketlendi!


In [99]:
# 1. Modeli Kaydet
model.save('llm_detector_model.h5')

import joblib
from tensorflow.keras.models import load_model

**Dosyaları yükle**

yuklenen_model = load_model('llm_detector_model.keras')
yuklenen_vect = joblib.load('vectorizer.pkl')

def hızlı_tahmin(metin):
    # 1. Temizle
    temiz_metin = clean_text(metin)
    # 2. Vektöre çevir (Sadece transform!)
    vektor = yuklenen_vect.transform([temiz_metin]).toarray()
    # 3. Tahmin et
    olasilik = yuklenen_model.predict(vektor)[0][0]
    
    sonuc = "Yapay Zeka" if olasilik > 0.5 else "İnsan"
    print(f"Sonuç: {sonuc} (Olasılık: %{olasilik*100:.2f})")

**Kullanımı:**

hızlı_tahmin(test3[0])

In [91]:
# Eğitimde kullandığımız temizleme fonksiyonunu test verisine uygulayalım
test['text'] = test['text'].apply(clean_text)

In [92]:
# Sadece transform işlemi yapılır
test_vect = vect.transform(test['text']).toarray()

In [93]:
# Modelden olasılıkları al
# Sigmoid aktivasyonu sayesinde değerler 0-1 arasındadır
predictions = model.predict(test_vect)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


In [94]:
# Teslimat dosyasını hazırla
submission = pd.DataFrame({
    'id': test['id'],
    'label': predictions.flatten() # Olasılık değerlerini tek boyutlu listeye çevirir
})

# Dosyayı kaydet
submission.to_csv('submission.csv', index=False)

print("Kaggle teslimat dosyası 'submission.csv' başarıyla oluşturuldu!")

Kaggle teslimat dosyası 'submission.csv' başarıyla oluşturuldu!


In [98]:
submission.head(10)

,id,label
0,0000aaaa,0.797644
1,1111bbbb,0.797644
2,2222cccc,0.797644


In [100]:
ornekler = ["i go school today collage is gud.", "The Electoral College is a complex political mechanism."]
ornek_vect = vect.transform(ornekler).toarray()
print(model.predict(ornek_vect))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
[[0.4284456]
 [0.8750882]]
